# Run fg2 — continuing the fg1 ladder from $\beta = 0.6657$ toward $\beta = 1$

Leg 2 of the full-$\gamma$-window Algorithm 1 run. Source: `partial_N8_m3_seed101_fg1.pt`
(job 776776, 106 sweeps, 7 levels, $\beta = 0.6657$, $m_{\mathrm{eff}} = 2.00$ of 3).

## Why this leg is worth running

fg1 did **not** stop because anything broke. It ran out of wall clock with the rejuvenation kernel
still healthy: level-mean acceptance $\bar{a} = 0.133$ at the last level and $0.221$ at the one before
(essentially the pCN target of $0.23$), `uniq/N = 1.00` at every level, `pair/iid` $\approx 1$, and
$s = 0.7139$ adapting normally, clear of both clamps. Novelty was still climbing monotonically:
$E_{q}[f]$ 1.0100 → 1.0565, $+3.2\sigma_{p}(f)$.

**Read fg1's Takeaways §5 before touching this.** The persisted `acc` column is a *single sweep*, not a
level average — `_record` is keyed by $\beta$ and overwritten every sweep. At $N = 8$ it is quantized
to eighths, and fg1's level-6 reading of `acc = 0.000` occurs **32% of the time** at that level's true
mean of 0.133. The first reading of fg1 mistook it for a collapsed kernel. It is not.

## The reparametrization, and the one thing it does not cover

$\pi_{\beta} \propto p \exp(\beta \lambda_{\max} f)$. With $\beta = \beta_{0} + (1 - \beta_{0})u$ this
is $\pi_{\beta_{0}} \exp(\lambda' u f)$ for $\lambda' = (1 - \beta_{0})\lambda_{\max}$, so the residual
leg is an ordinary SMC whose *starting distribution* is the saved cloud:

$$\beta_{0} = 0.6657, \quad \lambda_{\max} = 236.034, \quad \lambda_{\mathrm{base}} = \beta_{0}\lambda_{\max} = 157.123, \quad \lambda' = 78.911.$$

**THAT REPARAMETRIZATION IS EXACT FOR THE WEIGHTS AND ONLY FOR THE WEIGHTS.** `PCNKernel.step`'s
acceptance ratio $\exp(\beta \lambda (f' - f))$ is reversible w.r.t. the **latent Gaussian prior**, so
its base is *always* $p = G_{\#}N(0, I)$ — it has no way to represent $\pi_{\beta_{0}}$ as a base, and
nothing in the SMC loop tells it one exists. Passing $\lambda'$ straight to the library would target
$p \exp(u \cdot 78.911 \cdot f)$ while reporting tilts three times larger. That is failure (3), which
silently invalidated every number in run 2_1. The fix is `lam_base` inside the kernel; it cannot be
done by wrapping the call from outside, because the *recording* key must stay the SMC's own $\beta$.

## Configuration

| | |
|---|---|
| source | `partial_N8_m3_seed101_fg1.pt` — the **only** resumable artifact ($Z$ and $s$ live there; `levels_..._TIMEOUT.pt` has neither) |
| $\gamma$ window | $[2^{-10}, 2^{10}]$, unchanged — it is part of what $f$ *is*, so a resume onto a different window is meaningless |
| $\lambda_{s}$ | **78.678, read from the source config** — no re-probe, saving ~14 min |
| reward | $R = 64$, same `refs_R64_seed7.pt`, $N_{\gamma} = 30$, $N_{\epsilon} = 3$, seeds 7 / 123, `MAX_DENOISER_ROWS = 24` |
| kernel | `s = 0.7139` restored, `adapt_rate = 0.1` (the library default — 2_2 needed 0.25 only because *its* source had not persisted `s`) |
| budget | 8 h wall (`--time=480`), 1 h reserved |

## Two diagnostics fixed here

1. **Selection rule** now requires `stop_reason == 'decorr'` as well as diversity floors. fg1's rule
   picked level 6 — a level that stopped on `timeout` — because `uniq/N` and ESS/N looked perfect.
2. **Level-mean acceptance** is now tracked and persisted per level, instead of only the last sweep.


In [1]:
import faulthandler
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()

# Hang watchdog: dumps every thread's Python stack to the job's .err without killing the process, so a
# stall names its own frame. 30 min cadence -- any hang worth diagnosing outlives one interval, and a
# tighter cadence buries the SMC's diagnostics under tracebacks.
faulthandler.enable()
faulthandler.dump_traceback_later(1800, repeat=True, exit=False)

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
RUN_DIR = os.path.join(REPO, "notebooks", "flux_fullgamma_alg1")
os.makedirs(RUN_DIR, exist_ok=True)

# --- output tagging -------------------------------------------------------------------------------
# fg1 -> fg2 IS NOT COSMETIC. Output names derive from N / M_TILT / SEED, none of which change on a
# resume, so an untagged leg would rewrite partial_N8_m3_seed101_fg1.pt -- ITS OWN RESUME SOURCE -- on
# the very first sweep, and a later crash would leave nothing to fall back to. The assert in the SMC
# cell enforces src != dst. decoded_<tag>/ is also per-leg: this leg will produce a different number of
# levels than fg1's seven, so a shared directory would silently mix two runs' PNGs at different lambda.
RUN_TAG = "fg2"
SOURCE_CKPT = os.path.join(RUN_DIR, "partial_N8_m3_seed101_fg1.pt")   # read-only here
DECODED_DIR = os.path.join(RUN_DIR, f"decoded_{RUN_TAG}")
GRID_PNG = os.path.join(RUN_DIR, f"fullgamma_grid_{RUN_TAG}.png")

assert os.path.exists(SOURCE_CKPT), (
    f"missing {SOURCE_CKPT}. NOTE: levels_N8_m3_seed101_fg1_TIMEOUT.pt is NOT a substitute -- it "
    f"carries no aux/Z (the pCN chain state) and no s (the adapted step size).")

# --- wall clock -----------------------------------------------------------------------------------
# The SMC is one long library call and cannot be interrupted from outside; the kernel checks DEADLINE
# after every sweep instead. Under Slurm, SLURM_JOB_END_TIME tracks --time automatically.
#
# 8 h. Fixed costs are LOWER than fg1's because lambda_s is cached and the gamma-profile diagnostic is
# already done: ~10-25 min model load + ~23 min reference bank + ~3 min reward guard ~= 40 min, against
# fg1's 52. That leaves ~6.3 h ~ 105 sweeps. Finishing level 6 (see the SMC cell) may take 15-30 of
# them, so expect 2-3 further levels and beta ~ 0.85-0.95. beta = 1 IS NOT PROMISED -- every sweep is
# checkpointed and `s` is persisted, so a third leg resumes cleanly from partial_..._fg2.pt.
WALL_BUDGET_H = 8.0            # keep in sync with --time in flux_fullgamma_alg1_resume.slurm (480 min)
RESERVE_H = 1.0
try:
    _end = float(os.environ["SLURM_JOB_END_TIME"])
except (KeyError, ValueError):
    _end = None
if _end and _end > T_START:
    DEADLINE, WALL_BUDGET_H, _src = _end - RESERVE_H * 3600, (_end - T_START) / 3600, "SLURM"
else:
    DEADLINE, _src = T_START + (WALL_BUDGET_H - RESERVE_H) * 3600, "constant"

from creativity_measure import set_default_dtype

set_default_dtype(torch.float32)          # FLUX runs bf16; everything the SMC touches is fp32
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16

print(f"device {DEVICE} ({DTYPE})   budget {WALL_BUDGET_H:.2f} h via {_src}")
print(f"SMC deadline {time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
print(f"run tag {RUN_TAG!r}   RESUMING from {os.path.basename(SOURCE_CKPT)}")


device cuda (torch.bfloat16)   budget 7.93 h via SLURM
SMC deadline 23:36:36
run tag 'fg2'   RESUMING from partial_N8_m3_seed101_fg1.pt


In [2]:
from diffusers import FluxPipeline

MODEL_ID = "black-forest-labs/FLUX.1-dev"          # canonical id, kept for the CONFIG record
PROMPT = "A dog"

GUIDANCE = 1.5

IMG = 512
C, H, W = 16, IMG // 8, IMG // 8       # VAE downsamples 8x; FLUX latents have 16 channels
LATENT_DIM = C * H * W                 # 65536  (1024^2 would be 262144, and pCN mixes worse)

# --- loading is OFFLINE, as in fg1 -----------------------------------------------------------------
# The token in $HOME/.hf_token is stale: job 776480 died on a 401 GatedRepoError, and fg1's own
# preflight logged `hub access: UNAVAILABLE (GatedRepoError)` while running perfectly. Nothing here
# needs the hub -- the full 27-file snapshot is in $HF_HOME/hub, the slurm script stages it to
# node-local disk, and `from_pretrained` on a directory containing model_index.json never resolves a
# repo id. The slurm script exports HF_HUB_OFFLINE=1; the assert below is the real guard.
#
# THIS CELL MUST STAY BIT-IDENTICAL TO fg1's. The denoiser, G and the batch cap all feed f, and this
# leg is stitched onto fg1's ladder -- a change here would move f and invalidate the join.
MODEL_PATH = os.environ.get("FLUX_LOCAL_PATH") or MODEL_ID
if MODEL_PATH != MODEL_ID:
    assert os.path.isfile(os.path.join(MODEL_PATH, "model_index.json")), (
        f"FLUX_LOCAL_PATH={MODEL_PATH} has no model_index.json -- the slurm staging step did not "
        f"complete. Loading would fall back to the gated hub repo and 401.")
    print(f"loading from STAGED {MODEL_PATH} (offline; hub never contacted)", flush=True)
else:
    print("WARNING: FLUX_LOCAL_PATH unset -- falling back to the HF cache over NFS. This is the path "
          "that hung job 754633 for 6 h.", flush=True)

pipe = FluxPipeline.from_pretrained(MODEL_PATH, torch_dtype=DTYPE).to(DEVICE)
with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512
    )
# Free both text encoders (~9.5 GB, T5 dominates): the embeddings above are all we need.
pipe.text_encoder = pipe.text_encoder_2 = pipe.tokenizer = pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
transformer, vae = pipe.transformer.eval(), pipe.vae.eval()
VAE_SF, VAE_SHIFT = vae.config.scaling_factor, vae.config.shift_factor
print(f"loaded in {(time.time() - T_START) / 60:.1f} min; guidance={GUIDANCE}  "
      f"guidance_embeds={transformer.config.guidance_embeds}")
print(f"latents ({C}, {H}, {W}) = d {LATENT_DIM}  vae scaling={VAE_SF} shift={VAE_SHIFT}")


def decode(flat: torch.Tensor, chunk: int = 2) -> torch.Tensor:
    """Flat FLUX latents (B, d) -> images (B, 3, 512, 512) in [0, 1].

    Chunked because the VAE decoder, not the transformer, is the memory spike.
    """
    outs = []
    for i in range(0, flat.shape[0], chunk):
        lat = flat[i : i + chunk].reshape(-1, C, H, W).to(DEVICE, DTYPE) / VAE_SF + VAE_SHIFT
        with torch.no_grad():
            img = vae.decode(lat).sample
        outs.append(pipe.image_processor.postprocess(img.float(), output_type="pt").cpu())
    return torch.cat(outs)


loading from STAGED /tmp/arielzoizner/flux-snap-3de623fc (offline; hub never contacted)
loaded in 6.5 min; guidance=1.5  guidance_embeds=True
latents (16, 64, 64) = d 65536  vae scaling=0.3611 shift=0.1159


In [3]:
from creativity_measure.distances.edm_adapter import chunked_denoiser, edm_score_fn
from creativity_measure.generators.base import edm_generator

SIGMA_MIN, SIGMA_MAX = 2e-3, 80.0
N_STEPS = 8               # Heun ODE steps inside G (15 denoiser evals: 2/step, minus the last corrector)

# MEASURED on this card/model/resolution (job 695266): 24/48/96/128 rows ok, 192 OOM, s/row flat at
# ~0.25 across the range. R=64 with NUM_EPS=3 wants 192 rows for the reference-bank build, which is
# exactly the width that OOMs. 24 costs nothing in throughput and leaves headroom for an 8 h run.
#
# FIXED FOR THE RUN, and identical to fg1's. Chunking is mathematically identical but NOT bitwise
# (BLAS picks different kernels per batch width), so a different cap here would move f at round-off and
# the reward guard in the SMC cell would -- correctly -- refuse the resume.
MAX_DENOISER_ROWS = 24

# Both must be 2-D in diffusers 0.39; _prepare_latent_image_ids takes the *packed* (half) grid.
img_ids = FluxPipeline._prepare_latent_image_ids(1, H // 2, W // 2, DEVICE, DTYPE)
txt_ids = text_ids if text_ids.ndim == 2 else text_ids[0]


def denoiser(x: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """EDM denoiser D(x_sigma, sigma) = E[X | x_sigma] backed by the FLUX transformer.

    Flow matching interpolates, EDM adds:  x_t = (1-t)x_0 + t*eps  vs  x_sigma = x_0 + sigma*eps.
    Factoring out (1-t) makes them identical under  t = sigma/(1+sigma),  x_t = x_sigma/(1+sigma).
    FLUX predicts v = eps - x_0, so  x_t - t*v = x_0  exactly -- no approximation.

    The full gamma window [2^-10, 2^10] evaluates the score at sigma in [0.031, 32], i.e.
    t = sigma/(1+sigma) in [0.030, 0.970] -- both ends strictly inside the trained range.

    Returns fp32 (the transformer runs bf16): reward.x_refs pins dtype for the whole SMC, and bf16
    weight/ESS arithmetic would quietly corrupt the beta schedule.
    """
    b = x.shape[0]
    sig = sigma.reshape(b, 1, 1, 1).to(torch.float32)
    t = (sigma / (1.0 + sigma)).to(torch.float32)
    x_t = x.to(torch.float32) / (1.0 + sig)
    packed = FluxPipeline._pack_latents(x_t.to(DTYPE), b, C, H, W)
    with torch.no_grad():
        v = transformer(
            hidden_states=packed,
            timestep=t.to(DTYPE),                                   # forward() scales by 1000 itself
            guidance=torch.full((b,), GUIDANCE, device=x.device, dtype=torch.float32),
            pooled_projections=pooled_prompt_embeds.expand(b, -1).to(DTYPE),
            encoder_hidden_states=prompt_embeds.expand(b, -1, -1).to(DTYPE),
            txt_ids=txt_ids,
            img_ids=img_ids,
            return_dict=False,
        )[0]
    v = FluxPipeline._unpack_latents(v, IMG, IMG, 8).to(torch.float32)
    return (x_t - t.reshape(b, 1, 1, 1) * v).to(torch.float32)


# Wrap ONCE and build both consumers from the wrapper: covers the generator path and every score path,
# including the reference score_bank build, which is the widest call in the run.
denoiser_capped = chunked_denoiser(denoiser, MAX_DENOISER_ROWS)

G = edm_generator(denoiser_capped, img_shape=(C, H, W), sigma_min=SIGMA_MIN,
                  sigma_max=SIGMA_MAX, n_steps=N_STEPS)
score_fn = edm_score_fn(denoiser_capped, (C, H, W))
print(f"generator + score ready   sigma in [{SIGMA_MIN}, {SIGMA_MAX}]  n_steps={N_STEPS}  "
      f"denoiser cap {MAX_DENOISER_ROWS} rows")


generator + score ready   sigma in [0.002, 80.0]  n_steps=8  denoiser cap 24 rows


In [4]:
from creativity_measure import SquaredGlobalIEMDistance
from creativity_measure.tilt import NormalizedExpectedDistanceReward

# --- knobs: fg1's, unchanged. A resume is only valid if f is the same function it was ---------------
R, N_GAMMA, NUM_EPS = 64, 30, 3
N = 8
MAX_SWEEPS = 40
M_TILT = 3.0
ESS_TARGET = 0.80
SEED = 101

# --- load the source checkpoint --------------------------------------------------------------------
src = torch.load(SOURCE_CKPT, map_location="cpu", weights_only=False)
src_levels = src["levels"]
init_entry = src_levels[-1]                       # the deepest level reached: beta = 0.6657
BETA0 = float(init_entry["beta"])
LAM_FULL = float(src["config"]["lam_max"])        # 236.034, the lambda the FULL ladder targets
lam_s = float(src["config"]["lam_s"])             # 78.678 -- a property of the (unchanged) reward and p
LAM_BASE = BETA0 * LAM_FULL                       # 157.123 -- the tilt the incoming cloud ALREADY has
LAM_RESUME = (1.0 - BETA0) * LAM_FULL             # 78.911 -- the residual leg's tilt

# The reward identity. R / N_GAMMA / NUM_EPS / SEED are the usual suspects; GAMMA_LO is the one this
# family of runs added, and getting it wrong would silently target a different f while every other
# check passed.
_c = src["config"]
assert _c["R"] == R and _c["n_gamma"] == N_GAMMA and _c["num_eps"] == NUM_EPS and _c["seed"] == SEED
assert _c["max_denoiser_rows"] == MAX_DENOISER_ROWS, "denoiser chunking moved; f is not bitwise the same"
assert _c["prompt"] == PROMPT and _c["guidance"] == GUIDANCE and _c["n_steps"] == N_STEPS
assert _c.get("full_gamma_window"), "source is not a full-gamma-window run"
assert init_entry["X"].shape[0] == N, f"checkpoint holds {init_entry['X'].shape[0]} particles, N={N}"
assert "Z" in init_entry.get("aux", {}), (
    "checkpoint has no pCN latents; cannot resume a pCN chain. This is what distinguishes "
    "partial_*.pt from levels_*_TIMEOUT.pt -- use the former.")
assert "s" in init_entry, "checkpoint has no adapted step size"

# --- hardware identity: the check job 697271 did not have ------------------------------------------
# FLUX runs bf16 (8-bit mantissa, ~0.2% per-op resolution) and Ampere vs Ada select different
# tensor-core kernels and reduction orders, so identical denoiser inputs give different outputs at
# round-off. Through the score bank and the normalizer that shifted f by 11% of std_p(f) and correctly
# aborted 697271's resume. Assert here, ~2 min in, rather than after the ~23 min bank build.
GPU_NAME = torch.cuda.get_device_name(0)
SOURCE_GPU = _c["gpu"]
assert GPU_NAME == SOURCE_GPU, (
    f"resume scheduled on {GPU_NAME!r} but the source ran on {SOURCE_GPU!r}; bf16 FLUX does not "
    f"reproduce across architectures. Fix --constraint in flux_fullgamma_alg1_resume.slurm.")

print(f"source: {os.path.basename(SOURCE_CKPT)}   {len(src_levels)} levels, "
      f"{src['sweeps_done']} sweeps done")
print(f"  hardware: {GPU_NAME} -- matches the source run")
print(f"  resuming from level {len(src_levels) - 1}: beta0={BETA0:.4f}  "
      f"lam_eff={LAM_BASE:.2f}  m_eff={LAM_BASE / lam_s:.2f} of {M_TILT:.0f}")
print(f"  residual leg: lam' = (1 - {BETA0:.4f}) * {LAM_FULL:.2f} = {LAM_RESUME:.2f}")
print(f"  MH tilt per sweep = lam_base + u*lam' = {LAM_BASE:.2f} + u*{LAM_RESUME:.2f}")
print(f"  logw at resume: max|logw| = {float(init_entry['logw'].abs().max()):.3e} "
      f"(zeros => the cloud is post-resample and equally weighted, so starting at logw=0 is correct)")
print(f"  s at resume: {float(init_entry['s']):.4f}  (clamps are 1e-3 and 0.999, so this is free)")

# --- references: LOADED, never regenerated ---------------------------------------------------------
# Persisting the reference LATENTS takes G out of the reproducibility chain entirely, which is what
# makes the reward guard below able to pass at all (697271 regenerated them and drifted).
REFS_PATH = os.path.join(RUN_DIR, f"refs_R{R}_seed7.pt")
_rf = torch.load(REFS_PATH, map_location="cpu", weights_only=False)
latent_refs = _rf["x_refs"].to(DEVICE)
assert _rf["R"] == R and _rf["latent_dim"] == LATENT_DIM
assert _rf.get("gpu") == GPU_NAME, (
    f"refs were generated on {_rf.get('gpu')!r}, this job is on {GPU_NAME!r}; the latents reload "
    f"exactly but the score bank built from them would not match the source run")
S = latent_refs.std().item()

# --- the gamma window: fg1's, asserted against the source ------------------------------------------
GAMMA_LO = 2.0**-10
GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
assert abs(GAMMA_LO - _c["gamma_lo"]) < 1e-12 and abs(GAMMA_HI - _c["gamma_hi"]) < 1e-9, (
    f"gamma window moved: this leg [{GAMMA_LO:.6g}, {GAMMA_HI:.6g}] vs source "
    f"[{_c['gamma_lo']:.6g}, {_c['gamma_hi']:.6g}]. The window is part of what f IS -- a resume onto "
    f"a different one is meaningless, not merely inaccurate.")
GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)
GAMMA_LO_RUN2 = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)
print(f"\ngamma window [{GAMMA_LO:.4g}, {GAMMA_HI:.4g}] -- matches source. "
      f"(fg1 measured: readmitted band = {_c.get('frac_readmitted', float('nan')):.2%} of D^2_IEM)")

D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=123, score_fn=score_fn)
t0 = time.time()
reward = NormalizedExpectedDistanceReward(distance=D2, x_refs=latent_refs)   # builds the ref bank
t_norm = time.time() - t0
_bank_rows = (N_GAMMA - 1) * NUM_EPS * R
print(f"reference score bank rebuilt in {t_norm:.0f}s from {_bank_rows} rows "
      f"({t_norm / _bank_rows:.3f} s/row; fg1 measured 0.251)")

# Free normalization check: mean_r f(x_r) == (R-1)/R exactly (identity on the MEAN, not per element --
# see fg1 cell 4). Costs zero score rows: pairwise() aliases batch_scores to the cached bank.
_expect = (R - 1) / R
f_refs = reward(latent_refs)
print(f"f(refs): mean {float(f_refs.mean()):.8f}  expected {_expect:.8f}  "
      f"|dev| {abs(float(f_refs.mean()) - _expect):.2e}")
assert abs(float(f_refs.mean()) - _expect) < 1e-6, "normalizer/numerator disagree"

# --- lambda_s is CACHED, not re-measured -----------------------------------------------------------
# fg1 measured it on 32 held-out draws at a cost of 13.7 min. It is a property of the reward and p,
# both unchanged, and CLAUDE.md is explicit that lambda_s is an anchor rather than a measurement
# (+/-13% at n=32, and selection is on in-run uniq/N and ESS anyway). Re-probing would spend 14 min to
# re-derive a number we already have -- and, worse, a slightly different one, which would move
# LAM_FULL and make this leg target a different ladder than the levels it is stitched onto.
print(f"\nlambda_s = {lam_s:.3f} (CACHED from fg1; not re-probed)   "
      f"LAM_FULL = {M_TILT:.0f} * lambda_s = {LAM_FULL:.2f}")

# --- cost projection -------------------------------------------------------------------------------
GEN_ROWS = 2 * N_STEPS - 1
REW_ROWS = (N_GAMMA - 1) * NUM_EPS
S_ROW = 0.251                                  # fg1 measured 208.0 s/sweep vs 205.8 projected (1.1%)
t_sweep = N * (GEN_ROWS + REW_ROWS) * S_ROW
budget_s = (WALL_BUDGET_H - RESERVE_H) * 3600 - (time.time() - T_START)
sweeps_afford = budget_s / t_sweep
print(f"\nprojected {t_sweep / 60:.1f} min/sweep at {S_ROW} s/row   "
      f"budget left {budget_s / 3600:.2f} h -> ~{sweeps_afford:.0f} sweeps")
print(f"  fg1's effort per level: 12, 5, 7, 36, 30, 16 sweeps (level-mean acc 0.77 -> 0.13)")
print(f"  finishing level 6 will take some of that; at ~30 sweeps/level after, expect 2-3 levels")
for lbl, swl in (("optimistic (20 sweeps/level)", 20.0), ("likely (30/level, fg1's recent rate)", 30.0),
                 ("pessimistic (every level to the cap)", float(MAX_SWEEPS))):
    _lv = sweeps_afford / swl
    print(f"    {lbl:38s} {_lv:4.1f} levels -> beta ~ "
          f"{min(1.0, BETA0 + _lv * 0.10):.3f}  (fg1's recent d_beta ~ 0.10-0.12)")
print("  beta=1 is NOT promised. Every sweep is checkpointed and `s` is persisted, so a third leg "
      "resumes from partial_..._fg2.pt with the adapted step size intact.")


source: partial_N8_m3_seed101_fg1.pt   7 levels, 106 sweeps done
  hardware: NVIDIA RTX A6000 -- matches the source run
  resuming from level 6: beta0=0.6657  lam_eff=157.12  m_eff=2.00 of 3
  residual leg: lam' = (1 - 0.6657) * 236.03 = 78.91
  MH tilt per sweep = lam_base + u*lam' = 157.12 + u*78.91
  logw at resume: max|logw| = 0.000e+00 (zeros => the cloud is post-resample and equally weighted, so starting at logw=0 is correct)
  s at resume: 0.7139  (clamps are 1e-3 and 0.999, so this is free)

gamma window [0.0009766, 1024] -- matches source. (fg1 measured: readmitted band = 9.18% of D^2_IEM)
reference score bank rebuilt in 1527s from 5568 rows (0.274 s/row; fg1 measured 0.251)
f(refs): mean 0.98437500  expected 0.98437500  |dev| 0.00e+00

lambda_s = 78.678 (CACHED from fg1; not re-probed)   LAM_FULL = 3 * lambda_s = 236.03

projected 3.4 min/sweep at 0.251 s/row   budget left 6.39 h -> ~112 sweeps
  fg1's effort per level: 12, 5, 7, 36, 30, 16 sweeps (level-mean acc 0.77 -> 0.13

In [5]:
from creativity_measure import (LevelSnapshot, PCNKernel, RejuvenationStop,
                                adaptive_tempering_smc_sample)
# _State is private, but resuming is something the library does not (yet) expose. The alternative is
# calling PCNKernel.init and discarding its work -- a wasted G(z) + f(X) on 8 particles (~816 rows,
# ~3.5 min). Constructing the state directly is cheaper and states the intent plainly.
from creativity_measure.adaptive_tempering_smc import _State

# --- level-6 completion budget ---------------------------------------------------------------------
# fg1's stop_reason_history is ['decorr','decorr','decorr','decorr','decorr','timeout']: its LAST level
# did not finish rejuvenating -- the deadline fired inside kernel.step after 16 sweeps. Advancing beta
# from a partly-rejuvenated cloud bakes a short composite kernel into the middle of the stitched
# ladder, which is failure (4) of the 2_2 header. This leg finishes that level FIRST, at the source's
# own tilt, before beta is allowed to move.
#
# THE BASELINE IS GONE AND THIS OVERSHOOTS ON PURPOSE. `_record` is keyed by beta, so sweep 0 of that
# level was overwritten by sweep 16 and the post-resample origin the decorr metric wants is not on
# disk. The baseline used is the checkpointed cloud, i.e. a FULL fresh decorrelation is demanded on top
# of the 16 sweeps already done. Over-rejuvenating is the correct side to err on here.
#
# SIZING: fg1's level 6 ran at level-mean acceptance 0.133 with s = 0.7139, so the pCN refresh per
# sweep is roughly acc*(1 - sqrt(1 - s^2)) ~ 0.039 -- order 20-40 sweeps to decorrelate. The cap will
# probably bind before the budget does, and that is fine: it advances with finish_reason='cap', loudly.
FINISH_MAX_SWEEPS = 30
FINISH_BUDGET_H = 2.0        # hard ceiling; past this we advance anyway and say so
FINISH_STOP = RejuvenationStop(max_n_mcmc=FINISH_MAX_SWEEPS)   # min 4, decorr threshold 0.2

CONFIG = {
    "model_id": MODEL_ID, "img": IMG, "C": C, "H": H, "W": W, "latent_dim": LATENT_DIM,
    "vae_scaling_factor": VAE_SF, "vae_shift_factor": VAE_SHIFT,
    "N": N, "m_tilt": M_TILT, "lam_max": LAM_FULL, "lam_resume": LAM_RESUME, "lam_s": lam_s,
    "lam_base": LAM_BASE, "beta0": BETA0, "resumed_from": os.path.basename(SOURCE_CKPT),
    "run_tag": RUN_TAG, "ess_target": ESS_TARGET, "n_mcmc": None, "max_n_mcmc": MAX_SWEEPS,
    "seed": SEED, "prompt": PROMPT, "guidance": GUIDANCE, "n_steps": N_STEPS,
    "sigma_min": SIGMA_MIN, "sigma_max": SIGMA_MAX,
    "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS, "gamma_lo": GAMMA_LO, "gamma_hi": GAMMA_HI,
    "gamma_lo_run2": GAMMA_LO_RUN2, "full_gamma_window": True,
    "frac_readmitted": _c.get("frac_readmitted"), "data_scale_S": S,
    "max_denoiser_rows": MAX_DENOISER_ROWS, "s_row": S_ROW, "t_norm_s": t_norm, "gpu": GPU_NAME,
    "finish_max_sweeps": FINISH_MAX_SWEEPS, "finish_budget_h": FINISH_BUDGET_H, "adapt_rate": 0.1,
}
PARTIAL_CKPT = os.path.join(RUN_DIR, f"partial_N{N}_m{M_TILT:.0f}_seed{SEED}_{RUN_TAG}.pt")
assert os.path.abspath(PARTIAL_CKPT) != os.path.abspath(SOURCE_CKPT), \
    "destination equals the resume source -- the first sweep would destroy it"


class DeadlineReached(RuntimeError):
    """Raised by the kernel when the wall-clock deadline passes, to stop the SMC cleanly."""


class CheckpointingPCNKernel(PCNKernel):
    """Persists the cloud every sweep and stops the run at a wall-clock deadline.

    Keying by `beta` is the trick -- beta changes only between levels, so each entry is overwritten
    within a level and ends up holding that level's LAST sweep, i.e. exactly the post-rejuvenation
    cloud the final snapshot records. Writes are temp-file + os.replace, so a kill mid-write cannot
    corrupt the last good file.

    ================= `lam_base`: the failure-(3) fix, carried over from 2_2 =================
    pCN's proposal is reversible w.r.t. the LATENT GAUSSIAN PRIOR, so PCNKernel.step's acceptance ratio
    exp(beta*lam*(f' - f)) makes the sweep invariant to p * exp(beta*lam*f). Its base is ALWAYS
    p = G#N(0,I); it cannot represent pi_beta0 as a base no matter how the leg is reparametrized, and
    nothing in the SMC loop tells it one exists.
    `lam_base` is the tilt the incoming cloud ALREADY carries. PCNKernel only ever uses the PRODUCT
    beta*lam, so passing beta=1.0 with the full product is exact. The RECORDING key stays the SMC's own
    `beta`, which is why this cannot be done by wrapping the call from outside.
    NOT affected, deliberately: the SMC's reweighting. dbeta*lam' == dbeta_true*LAM_FULL exactly, and
    the dbeta bisection maps exactly too (ceiling 1.0 <-> 1-beta0).

    ===================== `acc_mean`: the fg1 reporting fix =====================
    fg1 persisted only the LAST sweep's acceptance per level. At N=8 that is quantized to eighths and
    wildly noisy -- its level-6 reading of 0.000 occurs 32% of the time at that level's true mean of
    0.133, and it was initially misread as a collapsed kernel. This tracks a running per-level mean
    alongside it, so the level's actual health is on disk rather than reconstructible only by inverting
    the Robbins-Monro recursion after the fact.
    """

    lam_base: float = 0.0        # 0.0 => bit-identical to a plain PCNKernel

    def __init__(self, *a, path, config, deadline, save_z=True, **kw):
        super().__init__(*a, **kw)
        self._path, self._config, self._deadline, self._save_z = path, config, deadline, save_z
        self._latest, self._sweeps_at, self._sweeps = {}, {}, 0
        self._acc_sum = {}                       # beta -> running sum of per-sweep acceptance

    def _record(self, state, *, beta, lam, acc, extra=None):
        n_at = self._sweeps_at.get(beta, 0)
        e = {"beta": beta, "lam_eff": beta * lam,
             "X": state.X.detach().to("cpu").clone(), "fX": state.fX.detach().to("cpu").clone(),
             "logw": state.logw.detach().to("cpu").clone(),
             "ess": float(getattr(state, "ess", float("nan"))),
             "s": float(self.s),
             "ancestors": torch.arange(state.X.shape[0]),
             "sweeps_at_beta": n_at, "acc": acc,
             # THE FIX: the level's mean acceptance so far, not just the last sweep's.
             "acc_mean": (self._acc_sum.get(beta, 0.0) / n_at) if n_at else float("nan")}
        if extra:
            e.update(extra)
        if self._save_z:
            e["aux"] = {k: v.detach().to("cpu").clone() for k, v in state.aux.items()}
        self._latest[beta] = e
        tmp = f"{self._path}.tmp"
        torch.save({"levels": list(self._latest.values()), "betas": list(self._latest.keys()),
                    "config": self._config, "partial": True, "sweeps_done": self._sweeps}, tmp)
        os.replace(tmp, self._path)

    def _mh_sweep(self, state, *, lam_eff, f, generator):
        """One pCN sweep whose invariant law is p*exp(lam_eff*f). No recording, no deadline check."""
        return PCNKernel.step(self, state, beta=1.0, lam=lam_eff, f=f, generator=generator)

    def init(self, n_particles, f, *, generator, device, dtype):
        st = super().init(n_particles, f, generator=generator, device=device, dtype=dtype)
        self._record(st, beta=0.0, lam=0.0, acc=float("nan"))
        return st

    def step(self, state, *, beta, lam, f, generator):
        acc = self._mh_sweep(state, lam_eff=self.lam_base + beta * lam, f=f, generator=generator)
        b = float(beta)
        self._sweeps += 1
        self._sweeps_at[b] = self._sweeps_at.get(b, 0) + 1
        self._acc_sum[b] = self._acc_sum.get(b, 0.0) + float(acc)
        self._record(state, beta=b, lam=float(lam), acc=float(acc))
        if time.time() > self._deadline:
            raise DeadlineReached(f"deadline after {self._sweeps} sweeps at beta={b:.4f}; "
                                  f"{len(self._latest)} levels on disk")
        return acc


class ResumePCNKernel(CheckpointingPCNKernel):
    """Starts the chain from a saved cloud, FINISHES that cloud's level, then hands it to the SMC.

    ---------------------------------- the reward guard ----------------------------------
    Correctness rests on the reward being the SAME function it was in fg1, so `init` recomputes f(X)
    and compares against the checkpoint's stored fX. One reward call (~696 rows, ~3 min) buys direct
    verification that the references, gamma window, Brownian bank and denoiser chunking all reproduced.

    THE TOLERANCE IS IN NATS OF LOG-WEIGHT, not fractions of std_p(f). Job 697271 used 1e-4 of std,
    which at std_p(f) = 0.0127 means |df| < 1.3e-6 -- three orders of magnitude below what bf16 can
    deliver on any hardware. What matters is the SPREAD of the drift, not its size: q is invariant
    under f -> f + c, so a uniform offset changes nothing while a spread distorts relative weights by
    up to exp(lam * spread). The spread is converted at LAM_FULL = 236.03, the top of the ladder, since
    with the corrected kernel the acceptance ratio runs at the TRUE lam_eff.

    MEAN_TOL is a separate tripwire on the offset. It cannot corrupt the target, but a shift past bf16
    round-off means a knob moved (wrong R, wrong gamma window, wrong references).
    """

    MEAN_TOL = 0.01                  # of |mean f|; bf16 round-off ~0.2%, a wrong knob is well past 1%

    def __init__(self, *a, init_entry, lam_base, lam_tol_scale, finish_stop, finish_deadline,
                 tol_nats=0.05, **kw):
        super().__init__(*a, **kw)
        self._init_entry, self._tol_nats = init_entry, tol_nats
        self.lam_base = float(lam_base)
        self._lam_tol_scale = float(lam_tol_scale)
        self._finish_stop, self._finish_deadline = finish_stop, finish_deadline
        self.finish_sweeps, self.finish_reason = 0, "not-run"
        self.finish_acc, self.finish_decorr = float("nan"), float("nan")
        self.f_join_before, self.f_join_after = float("nan"), float("nan")

    def _check_reward(self, fX, n_particles):
        e = self._init_entry
        fX_saved = e["fX"].to(device=fX.device, dtype=fX.dtype)
        d = fX - fX_saved
        f_mean = float(fX_saved.mean())
        offset = float(d.mean())                         # cancels in q; tripwire only
        spread = float(d.max() - d.min())                # this is what moves the relative weights
        nats = self._lam_tol_scale * spread
        print(f"reward reproducibility on {n_particles} particles:\n"
              f"  offset {offset:+.3e} ({offset / f_mean:+.3%} of f)   "
              f"max|df| {float(d.abs().max()):.3e} "
              f"({float(d.abs().max()) / float(fX_saved.std().clamp_min(1e-12)):.2%} of std_q(f))\n"
              f"  spread {spread:.3e} -> {nats:.4f} nats at LAM_FULL={self._lam_tol_scale:.1f} "
              f"(tolerance {self._tol_nats})")
        if nats > self._tol_nats:
            raise RuntimeError(
                f"reward did not reproduce: {nats:.3f} nats of log-weight distortion > "
                f"{self._tol_nats}. The resumed leg would target a different distribution than the "
                f"levels it is stitched onto. First suspect is the GPU -- bf16 FLUX does not "
                f"reproduce across architectures (this killed 697271); then the gamma window, then "
                f"R / N_GAMMA / NUM_EPS / seeds / MAX_DENOISER_ROWS.")
        if abs(offset) > self.MEAN_TOL * abs(f_mean):
            raise RuntimeError(
                f"reward offset {offset / f_mean:+.2%} of f exceeds {self.MEAN_TOL:.0%}. A uniform "
                f"shift does not change q, but it is far past bf16 round-off (~0.2%), so a reward "
                f"knob has moved. Check the gamma window first -- it is the one this run family "
                f"varies -- then R / N_GAMMA / NUM_EPS / the reference set.")

    def _finish_level(self, state, f, generator):
        """Sweep at the SOURCE tilt until the level fg1 left on `timeout` decorrelates."""
        stop = self._finish_stop
        print(f"\nfinishing fg1's last level at lam_eff = {self.lam_base:.2f} "
              f"(it stopped on 'timeout' after {self._init_entry.get('sweeps_at_beta', '?')} sweeps, "
              f"not on 'decorr')")
        print(f"  baseline = the CHECKPOINTED cloud (sweep 0 of that level is not on disk), so this "
              f"demands a full fresh decorrelation -- deliberately more than the rule asked for")
        print(f"  cap {stop.max_n_mcmc} sweeps / budget until "
              f"{time.strftime('%H:%M:%S', time.localtime(self._finish_deadline))}, "
              f"s0={self.s:.4f} adapt_rate={self.adapt_rate}")
        baseline = self.snapshot_baseline(state)
        accs, reason = [], "cap"
        for t in range(stop.max_n_mcmc):
            if time.time() > min(self._finish_deadline, self._deadline):
                reason = "budget"
                break
            accs.append(self._mh_sweep(state, lam_eff=self.lam_base, f=f, generator=generator))
            self._sweeps += 1
            self._sweeps_at[0.0] = self._sweeps_at.get(0.0, 0) + 1
            self._acc_sum[0.0] = self._acc_sum.get(0.0, 0.0) + float(accs[-1])
            self.finish_decorr = float(self.decorrelation(state, baseline))
            # lam=0.0 => lam_eff=0.0 on the record; the stitch cell adds BETA0*LAM_FULL back, so this
            # row lands at the true 157.12. Written every sweep, so a kill keeps the progress.
            self._record(state, beta=0.0, lam=0.0, acc=accs[-1],
                         extra={"finish_decorr": self.finish_decorr, "lam_eff_mh": self.lam_base})
            _run = sum(accs) / len(accs)
            print(f"    finish sweep {t + 1:2d}/{stop.max_n_mcmc}  acc={accs[-1]:.3f}  "
                  f"mean={_run:.3f}  decorr={self.finish_decorr:.3f}  s={self.s:.4f}  "
                  f"E[f]={float(state.fX.mean()):.4f}", flush=True)
            if t >= stop.min_n_mcmc - 1 and self.finish_decorr < stop.decorr_threshold:
                reason = "decorr"
                break
        self.finish_sweeps = len(accs)
        self.finish_reason = reason
        self.finish_acc = sum(accs) / len(accs) if accs else float("nan")
        tag = {"decorr": "OK", "cap": "!! HIT THE SWEEP CAP", "budget": "!! RAN OUT OF BUDGET"}[reason]
        print(f"  -> {self.finish_sweeps} sweeps, stop='{reason}' ({tag}), "
              f"mean acc {self.finish_acc:.3f}, decorr {self.finish_decorr:.3f}, s={self.s:.4f}")
        print(f"     E[f] at the join: {self.f_join_before:.4f} -> {float(state.fX.mean()):.4f}")
        if reason != "decorr":
            print("  *** WARNING: level 6 is STILL not fully rejuvenated. The stitched ladder keeps\n"
                  "  *** fg1's defect, smaller. Say so in the Takeaways and treat the join as a lower\n"
                  "  *** bound on mixing, not as clean draws from q.")

    def init(self, n_particles, f, *, generator, device, dtype):
        e = self._init_entry
        X = e["X"].to(device=device, dtype=dtype)
        Z = e["aux"]["Z"].to(device=device, dtype=dtype)
        fX = f(X)                                        # recompute, do not trust the stored value
        self._check_reward(fX, n_particles)
        # logw = 0: the source cloud is post-resample and equally weighted (asserted in cell 4).
        st = _State(X=X, fX=fX, logw=torch.zeros(n_particles, device=device, dtype=dtype),
                    aux={"Z": Z})
        self.f_join_before = float(fX.mean())
        self._record(st, beta=0.0, lam=0.0, acc=float("nan"))   # the un-finished join, for the record
        self._finish_level(st, f, generator)
        self.f_join_after = float(st.fX.mean())
        return st


kernel = ResumePCNKernel(
    G, LATENT_DIM, init_entry=init_entry,
    lam_base=LAM_BASE,                  # 157.12 -- THE FIX: the tilt the incoming cloud already has
    lam_tol_scale=LAM_FULL,             # the guard's nats are quoted at the top of the ladder
    finish_stop=FINISH_STOP,
    finish_deadline=T_START + FINISH_BUDGET_H * 3600,
    path=PARTIAL_CKPT, config=CONFIG, deadline=DEADLINE,
)
# fg1 DID persist `s`, unlike the source 2_2 had to work with -- so this leg keeps the adapted step
# size instead of restarting from s0 = 0.4 and re-learning the geometry. That is also why adapt_rate
# stays at the library default of 0.1; 2_2's 0.25 was a workaround for exactly this state being absent.
kernel.s = float(init_entry["s"])
print(f"restored adapted pCN step size s = {kernel.s:.4f} "
      f"(s0 default is 0.400; clamps are 1e-3 / 0.999)")

print(f"\nSMC (resumed): N={N} R={R} N_gamma={N_GAMMA} beta0={BETA0:.4f} "
      f"lam'={LAM_RESUME:.1f} (of full {LAM_FULL:.1f})  lam_base={LAM_BASE:.2f}  "
      f"ess_target={ESS_TARGET} n_mcmc=None (adaptive, cap {MAX_SWEEPS})")
print(f"projected {t_sweep / 60:.1f} min/sweep; deadline "
      f"{time.strftime('%H:%M:%S', time.localtime(DEADLINE))}", flush=True)

t0, timed_out = time.time(), False
try:
    res = adaptive_tempering_smc_sample(
        reward, lam=LAM_RESUME, n_particles=N, kernel=kernel, ess_target=ESS_TARGET,
        n_mcmc=None, max_n_mcmc=MAX_SWEEPS, keep_levels=True, snapshot_device="cpu", seed=SEED,
    )
    levels = res.levels
    assert levels is not None
except DeadlineReached as exc:
    timed_out = True
    print(f"\n*** STOPPED ON DEADLINE: {exc}\n*** Rendering the levels that completed.")
    _e = list(kernel._latest.values())
    levels = [LevelSnapshot(beta=x["beta"], lam_eff=x["lam_eff"], X=x["X"], fX=x["fX"],
                            logw=x["logw"], ancestors=x["ancestors"],
                            ess=x.get("ess", float("nan"))) for x in _e]

    class _PartialResult:
        """Stands in for the result dataclass. ESS is read back from the kernel's per-level records
        (`state.ess`, set by the SMC loop before resampling). Stop reasons are INFERRED: all sweeps
        used -> 'cap', fewer -> 'decorr', and the level in progress at the deadline -> 'timeout'."""

        def __init__(s, entries):
            tail = entries[1:]
            s.betas = [x["beta"] for x in tail]
            s.ess_history = [x.get("ess", float("nan")) for x in tail]
            s.acc_history = [x["acc"] for x in tail]
            s.acc_mean_history = [x.get("acc_mean", float("nan")) for x in tail]
            s.n_mcmc_history = [x["sweeps_at_beta"] for x in tail]
            s.stop_reason_history = [("cap" if x["sweeps_at_beta"] >= MAX_SWEEPS else "decorr")
                                     for x in tail]
            if s.stop_reason_history:
                s.stop_reason_history[-1] = "timeout"
            s.X, s.logw, s.levels = levels[-1].X, levels[-1].logw, levels

    res = _PartialResult(_e)

# On the non-timeout path the library result has no acc_mean; read it back from the kernel's records.
if not hasattr(res, "acc_mean_history"):
    _by_beta = {round(k, 10): v.get("acc_mean", float("nan")) for k, v in kernel._latest.items()}
    res.acc_mean_history = [_by_beta.get(round(b, 10), float("nan")) for b in res.betas]

mins = (time.time() - t0) / 60
ckpt = os.path.join(RUN_DIR,
                    f"levels_N{N}_m{M_TILT:.0f}_seed{SEED}_{RUN_TAG}"
                    f"{'_TIMEOUT' if timed_out else ''}.pt")
torch.save({
    "levels": [{"beta": s.beta, "lam_eff": s.lam_eff, "X": s.X, "fX": s.fX,
                "logw": s.logw, "ancestors": s.ancestors, "ess": s.ess} for s in levels],
    "X": res.X, "logw": res.logw, "betas": res.betas,
    "ess_history": res.ess_history, "acc_history": res.acc_history,
    "acc_mean_history": res.acc_mean_history,
    "n_mcmc_history": res.n_mcmc_history, "stop_reason_history": res.stop_reason_history,
    "partial": timed_out,
    "config": {**CONFIG, "wall_clock_min": mins, "timed_out": timed_out,
               "finish_sweeps": kernel.finish_sweeps, "finish_reason": kernel.finish_reason,
               "finish_acc": kernel.finish_acc, "finish_decorr": kernel.finish_decorr,
               "f_join_before": kernel.f_join_before, "f_join_after": kernel.f_join_after,
               "s_final": float(kernel.s)},
}, ckpt)
print(f"\n{'TIMED OUT' if timed_out else 'completed'} in {mins:.1f} min ({mins / 60:.2f} h)   "
      f"levels={len(levels)} (this leg)")
print(f"level-6 completion: {kernel.finish_sweeps} sweeps, stop='{kernel.finish_reason}', "
      f"E[f] {kernel.f_join_before:.4f} -> {kernel.f_join_after:.4f}")
print(f"saved -> {os.path.basename(ckpt)}  ({os.path.getsize(ckpt) / 2**20:.1f} MB)")
print(f"actual vs projected sweep cost: "
      f"{mins * 60 / max(1, sum(res.n_mcmc_history) + kernel.finish_sweeps):.1f} s vs {t_sweep:.1f} s")
print(f"beta reached (TRUE): {BETA0 + levels[-1].beta * (1 - BETA0):.4f} of 1.0   "
      f"s_final={float(kernel.s):.4f}")


restored adapted pCN step size s = 0.7139 (s0 default is 0.400; clamps are 1e-3 / 0.999)

SMC (resumed): N=8 R=64 N_gamma=30 beta0=0.6657 lam'=78.9 (of full 236.0)  lam_base=157.12  ess_target=0.8 n_mcmc=None (adaptive, cap 40)
projected 3.4 min/sweep; deadline 23:36:36
reward reproducibility on 8 particles:
  offset +0.000e+00 (+0.000% of f)   max|df| 0.000e+00 (0.00% of std_q(f))
  spread 0.000e+00 -> 0.0000 nats at LAM_FULL=236.0 (tolerance 0.05)

finishing fg1's last level at lam_eff = 157.12 (it stopped on 'timeout' after 16 sweeps, not on 'decorr')
  baseline = the CHECKPOINTED cloud (sweep 0 of that level is not on disk), so this demands a full fresh decorrelation -- deliberately more than the rule asked for
  cap 30 sweeps / budget until 18:40:40, s0=0.7139 adapt_rate=0.1
    finish sweep  1/30  acc=0.000  mean=0.000  decorr=1.000  s=0.6977  E[f]=1.0565
    finish sweep  2/30  acc=0.125  mean=0.062  decorr=0.965  s=0.6904  E[f]=1.0580
    finish sweep  3/30  acc=0.125  mean=0.0

In [6]:
# --- stitch: fg1's ladder + this leg, everything remapped to TRUE beta / lambda --------------------
# This leg's SMC reports beta as u in [0, 1] over the RESIDUAL interval, so every row must be remapped:
#     beta_true    = BETA0 + u * (1 - BETA0)
#     lam_eff_true = LAM_BASE + u * LAM_RESUME   ( == beta_true * LAM_FULL )
# Without that, every row of this leg reads low by m_eff = 2.00, i.e. two thirds of the tilt.
#
# This leg's level 0 (u = 0) is the FINISHED version of fg1's level 6, so it REPLACES fg1's last row
# rather than sitting next to it. Provenance is in the `src` column so the join is never ambiguous.
FG1_LEVELS_PT = os.path.join(RUN_DIR, "levels_N8_m3_seed101_fg1_TIMEOUT.pt")
fg1 = torch.load(FG1_LEVELS_PT, map_location="cpu", weights_only=False)

rows = []
for k, L in enumerate(fg1["levels"][:-1]):          # fg1 levels 0..5; level 6 is superseded below
    rows.append({
        "src": "fg1", "k": k, "beta": L["beta"], "lam_eff": L["lam_eff"],
        "fX": L["fX"], "X": L["X"], "ess": (fg1["ess_history"][k - 1] / N) if k else float("nan"),
        "acc": (fg1["acc_history"][k - 1] if k else float("nan")),
        "swp": (fg1["n_mcmc_history"][k - 1] if k else 0),
        "stop": (fg1["stop_reason_history"][k - 1] if k else "-"),
        # fg1 stored only the LAST sweep's acceptance. Recover the level mean by inverting the
        # Robbins-Monro recursion  log s_{t+1} = log s_t + eta (a_t - a*)  over the level's sweeps.
        # Levels whose s touched the 0.999 ceiling are contaminated (clamping discards upward
        # movement), so those means are lower bounds and are flagged.
        "acc_mean": float("nan"), "clamped": False,
    })
for k in range(1, len(rows)):
    s0, s1 = src_levels[k - 1]["s"], src_levels[k]["s"]
    n_sw = src_levels[k]["sweeps_at_beta"]
    if n_sw:
        rows[k]["acc_mean"] = 0.23 + (math.log(s1) - math.log(s0)) / (0.1 * n_sw)
        rows[k]["clamped"] = (s0 >= 0.9989 or s1 >= 0.9989)

for k, L in enumerate(levels):                       # this leg, u -> true
    u = L.beta
    rows.append({
        "src": RUN_TAG, "k": k, "beta": BETA0 + u * (1.0 - BETA0),
        "lam_eff": LAM_BASE + u * LAM_RESUME,
        "fX": L.fX, "X": L.X,
        "ess": (res.ess_history[k - 1] / N) if k else float("nan"),
        "acc": (res.acc_history[k - 1] if k else kernel.finish_acc),
        "acc_mean": (res.acc_mean_history[k - 1] if k else kernel.finish_acc),
        "swp": (res.n_mcmc_history[k - 1] if k else kernel.finish_sweeps),
        "stop": (res.stop_reason_history[k - 1] if k else kernel.finish_reason),
        "clamped": False,
    })

Eqf = [float(r["fX"].mean()) for r in rows]
f_sd = [float(r["fX"].std()) for r in rows]
uniq = [int(r["X"].unique(dim=0).shape[0]) / N for r in rows]
_ref_pair = float(torch.pdist(latent_refs.cpu().double()).median())
pair = [float(torch.pdist(r["X"].cpu().double()).median()) / _ref_pair for r in rows]
base_f = Eqf[0]

print(f"gamma window [{GAMMA_LO:.4g}, {GAMMA_HI:.4g}]  lambda_s={lam_s:.2f}  "
      f"LAM_FULL={LAM_FULL:.2f}  (fg1 + {RUN_TAG} stitched)")
print(f"level-6 completion: {kernel.finish_sweeps} sweeps at lam_eff={LAM_BASE:.1f}, "
      f"stop='{kernel.finish_reason}', mean acc {kernel.finish_acc:.3f}, "
      f"decorr {kernel.finish_decorr:.3f}")
print(f"this leg: betas(u)={[round(b, 4) for b in res.betas]}  "
      f"n_mcmc={res.n_mcmc_history}  stops={res.stop_reason_history}")
print(f"\n{'src':>4} {'k':>2} {'beta':>7} {'lam_eff':>9} {'m':>6} {'E_q[f]':>8} {'d f/sd':>7} "
      f"{'ESS/N':>6} {'acc*':>6} {'accbar':>7} {'uniq/N':>7} {'pair':>6} {'f-sd':>7} "
      f"{'swp':>4} {'stop':>8}")
for i, r in enumerate(rows):
    flag = "!" if r["clamped"] else " "
    print(f"{r['src']:>4} {r['k']:2d} {r['beta']:7.4f} {r['lam_eff']:9.1f} "
          f"{r['lam_eff'] / lam_s:6.2f} {Eqf[i]:8.4f} {(Eqf[i] - base_f) / f_sd[0]:7.1f} "
          f"{r['ess']:6.2f} {r['acc']:6.2f} {r['acc_mean']:6.3f}{flag} {uniq[i]:7.2f} "
          f"{pair[i]:6.2f} {f_sd[i]:7.4f} {r['swp']:4d} {r['stop']:>8}")
print("  acc* = LAST sweep only (quantized to 1/N=0.125, very noisy). accbar = level MEAN "
      "-- read this one.")
print("  ! = s touched the 0.999 clamp during that level, so accbar is a lower bound.")

print(f"\nnovelty: E_q[f] {base_f:.4f} -> {Eqf[-1]:.4f} "
      f"({100 * (Eqf[-1] / base_f - 1):+.1f}%, {(Eqf[-1] - base_f) / f_sd[0]:.1f} sigma_p(f))")
print(f"beta reached (TRUE): {rows[-1]['beta']:.4f} of 1.0   "
      f"m_eff {rows[-1]['lam_eff'] / lam_s:.2f} of {M_TILT:.0f}   "
      f"(fg1 ended at {BETA0:.4f} / {LAM_BASE / lam_s:.2f})")

# --- selection, WITH THE fg1 FIX -------------------------------------------------------------------
# fg1's rule keyed only on uniq/N and ESS/N and therefore selected a level that had stopped on
# 'timeout' -- under-rejuvenated, and looking perfect precisely BECAUSE its rejuvenation was cut short.
# A level that did not meet its own convergence rule is not a draw from q, however healthy it looks.
_ok = [i for i in range(1, len(rows))
       if uniq[i] >= 0.5 and rows[i]["ess"] >= 0.5 and rows[i]["stop"] == "decorr"]
if _ok:
    i = _ok[-1]
    print(f"\nSELECTED {rows[i]['src']} level {rows[i]['k']}: lam_eff={rows[i]['lam_eff']:.1f} "
          f"(m={rows[i]['lam_eff'] / lam_s:.2f}), E_q[f]={Eqf[i]:.4f}, uniq/N={uniq[i]:.2f}, "
          f"ESS/N={rows[i]['ess']:.2f}, stop='decorr', accbar={rows[i]['acc_mean']:.3f}")
else:
    print("\nNO level satisfies uniq/N >= 0.5 AND ESS/N >= 0.5 AND stop == 'decorr'. Report that.")
_excl = [i for i in range(1, len(rows))
         if uniq[i] >= 0.5 and rows[i]["ess"] >= 0.5 and rows[i]["stop"] != "decorr"]
if _excl:
    print(f"  EXCLUDED by the stop_reason rule (would have been selected under fg1's rule): "
          f"{[(rows[i]['src'], rows[i]['k'], rows[i]['stop']) for i in _excl]}")

# --- diagnostics ----------------------------------------------------------------------------------
lam_eff = [r["lam_eff"] for r in rows]
n_fg1 = sum(1 for r in rows if r["src"] == "fg1")
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
ax[0].plot(lam_eff, Eqf, "o-"); ax[0].set_ylabel(r"$E_q[f]$"); ax[0].set_title("novelty vs tilt")
ax[1].plot(lam_eff, uniq, "o-", label="uniq/N")
ax[1].plot(lam_eff, [r["ess"] for r in rows], "s--", label="ESS/N")
ax[1].plot(lam_eff, pair, "^-", label="pair/iid")
ax[1].axhline(0.5, color="crimson", ls=":", label="0.5 floor")
ax[1].set_ylim(0, 1.15); ax[1].legend(fontsize=8); ax[1].set_title("diversity")
ax[2].plot(lam_eff, [r["acc_mean"] for r in rows], "o-", color="darkorange", label="level mean")
ax[2].plot(lam_eff, [r["acc"] for r in rows], "x:", color="grey", alpha=0.6, label="last sweep")
ax[2].axhline(0.23, color="grey", ls=":", label="pCN target")
ax[2].set_ylabel("acceptance"); ax[2].legend(fontsize=8); ax[2].set_title("rejuvenation health")
ax[3].plot(lam_eff, f_sd, "o-", color="darkgreen")
ax[3].set_ylabel(r"std$_q(f)$"); ax[3].set_title("f-spread")
for a in ax:
    a.axvline(LAM_BASE, color="purple", ls="--", lw=1, alpha=0.7)
    a.set_xlabel(r"$\lambda_{eff}$"); a.grid(alpha=0.3)
ax[0].text(LAM_BASE, ax[0].get_ylim()[0], "  fg1|fg2 join", color="purple", fontsize=7, va="bottom")
plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, f"diagnostics_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
plt.show()


gamma window [0.0009766, 1024]  lambda_s=78.68  LAM_FULL=236.03  (fg1 + fg2 stitched)
level-6 completion: 24 sweeps at lam_eff=157.1, stop='budget', mean acc 0.094, decorr 0.698
this leg: betas(u)=[0.435, 0.7071]  n_mcmc=[40, 40]  stops=['cap', 'timeout']

 src  k    beta   lam_eff      m   E_q[f]  d f/sd  ESS/N   acc*  accbar  uniq/N   pair    f-sd  swp     stop
 fg1  0  0.0000       0.0   0.00   1.0100     0.0    nan    nan    nan     1.00   0.97  0.0145    0        -
 fg1  1  0.1322      31.2   0.40   1.0083    -0.1   0.80   0.75  0.771     1.00   0.98  0.0208   12   decorr
 fg1  2  0.2342      55.3   0.70   1.0106     0.0   0.80   0.50  0.650     1.00   0.92  0.0168    5   decorr
 fg1  3  0.3438      81.1   1.03   1.0287     1.3   0.80   0.50  0.310!    1.00   0.94  0.0234    7   decorr
 fg1  4  0.4463     105.3   1.34   1.0322     1.5   0.80   0.38  0.188!    1.00   1.01  0.0187   36   decorr
 fg1  5  0.5677     134.0   1.70   1.0427     2.3   0.80   0.25  0.221     1.00   1.02  0

In [7]:
# --- images: the FULL stitched ladder, both legs in one grid ---------------------------------------
# Lineage ordering does NOT survive the join: fg1's ancestors and this leg's index different resampling
# histories, and the checkpointed rows carry `arange` anyway (resampling happens outside the kernel).
# So columns are raw particle index throughout, and the join is marked instead of pretended away.
# Within this leg the ordering IS lineage-consistent for the SMC-returned levels.
decoded = [decode(r["X"]) for r in rows]
print(f"decoded {len(rows)} levels x {N} particles")

nrow = len(rows)
fig, axes = plt.subplots(nrow, N, figsize=(N * 1.9, nrow * 2.05), squeeze=False)
for k in range(nrow):
    for j in range(N):
        a = axes[k][j]
        a.axis("off")
        a.imshow(decoded[k][j].permute(1, 2, 0).clamp(0, 1).numpy())
        if j == 0:
            a.text(-0.16, 0.5, f"[{rows[k]['src']}] $\\lambda_{{eff}}$={rows[k]['lam_eff']:.0f}\n"
                               f"$\\beta$={rows[k]['beta']:.3f}  m={rows[k]['lam_eff'] / lam_s:.2f}\n"
                               f"$E_q[f]$={Eqf[k]:.3f}",
                   transform=a.transAxes, ha="right", va="center", fontsize=7)
        if k == 0:
            a.set_title(f"col {j}", fontsize=7)
        # Mark the leg boundary so no one reads a lineage across it.
        if rows[k]["src"] == RUN_TAG and (k == 0 or rows[k - 1]["src"] == "fg1"):
            for sp in a.spines.values():
                sp.set_visible(True); sp.set_color("purple"); sp.set_linewidth(2)
fig.suptitle(f"FULL $\\gamma$ window $[2^{{-10}}, 2^{{10}}]$, Alg 1 -- fg1 + {RUN_TAG} stitched\n"
             f"$\\lambda_s$={lam_s:.1f}, $\\lambda_{{max}}$={LAM_FULL:.0f} (m={M_TILT:.0f}), "
             f"prompt {PROMPT!r}, N={N}   purple = leg boundary at $\\beta$={BETA0:.4f}",
             fontsize=12)
plt.tight_layout(rect=(0.06, 0, 1, 0.965))
fig.savefig(GRID_PNG, dpi=110, bbox_inches="tight")
plt.show()

# Untilted anchor, fg1's end, and this leg's top -- the three rows worth looking at side by side.
_marks = [(0, "UNTILTED $\\lambda_{eff}=0$ (samples from $p$)"),
          (n_fg1 - 1, f"fg1's LAST COMPLETED level, $\\lambda_{{eff}}$={rows[n_fg1 - 1]['lam_eff']:.0f}"),
          (len(rows) - 1, f"{RUN_TAG} TOP, $\\lambda_{{eff}}$={lam_eff[-1]:.0f} "
                          f"(m={lam_eff[-1] / lam_s:.2f})")]
for k, label in _marks:
    fig, axes = plt.subplots(1, N, figsize=(N * 1.9, 2.6), squeeze=False)
    for j in range(N):
        a = axes[0][j]
        a.axis("off")
        a.imshow(decoded[k][j].permute(1, 2, 0).clamp(0, 1).numpy())
        a.set_title(f"f={float(rows[k]['fX'][j]):.4f}", fontsize=7)
    fig.suptitle(label, fontsize=12)
    plt.tight_layout(rect=(0, 0, 1, 0.90))
    fig.savefig(os.path.join(RUN_DIR, f"level{k:02d}_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
    plt.show()

# Per-image PNGs for full-resolution inspection, tagged by leg so the directory cannot mix runs.
from PIL import Image

os.makedirs(DECODED_DIR, exist_ok=True)
for k in range(len(rows)):
    for j in range(N):
        arr = (decoded[k][j].permute(1, 2, 0).clamp(0, 1) * 255).round().to(torch.uint8).numpy()
        Image.fromarray(arr).save(os.path.join(
            DECODED_DIR, f"{rows[k]['src']}_lvl{k:02d}_lam{rows[k]['lam_eff']:.0f}_col{j}.png"))
print(f"wrote {len(rows) * N} PNGs to {os.path.basename(DECODED_DIR)}/ and "
      f"{os.path.basename(GRID_PNG)}")

# Latent-space health. A tilt that pushes particles off the manifold inflates the latent norm, and the
# VAE only decodes sensibly near the scale it was trained at. fg1 saw these DRIFT DOWN (0.979 -> 0.937
# vs references at 1.0147), which is what ruled out the Euclidean-blow-out failure mode. Watch whether
# the trend continues or reverses now that lambda_eff goes past 157.
print(f"\nlatent std by level (references: {S:.4f}):")
for k, r in enumerate(rows):
    print(f"  [{r['src']:>3}] lvl {k:2d}  lam_eff={r['lam_eff']:8.1f}  "
          f"std={float(r['X'].std()):.4f}  mean|x|={float(r['X'].abs().mean()):.4f}")


decoded 9 levels x 8 particles
wrote 72 PNGs to decoded_fg2/ and fullgamma_grid_fg2.png

latent std by level (references: 1.0147):
  [fg1] lvl  0  lam_eff=     0.0  std=0.9793  mean|x|=0.7699
  [fg1] lvl  1  lam_eff=    31.2  std=0.9887  mean|x|=0.7829
  [fg1] lvl  2  lam_eff=    55.3  std=0.9217  mean|x|=0.7304
  [fg1] lvl  3  lam_eff=    81.1  std=0.9709  mean|x|=0.7670
  [fg1] lvl  4  lam_eff=   105.3  std=0.9626  mean|x|=0.7589
  [fg1] lvl  5  lam_eff=   134.0  std=0.9636  mean|x|=0.7624
  [fg2] lvl  6  lam_eff=   157.1  std=0.9019  mean|x|=0.7164
  [fg2] lvl  7  lam_eff=   191.4  std=0.9053  mean|x|=0.7175
  [fg2] lvl  8  lam_eff=   212.9  std=0.9989  mean|x|=0.7960


## Takeaways

*(To be written after the run. Per the repo convention this cell is the authoritative statement of what
was learned; the tables above are evidence, not conclusions.)*

Read in this order:

1. **`finish_reason` first.** If level 6 did not reach `decorr`, the stitched ladder still carries fg1's
   defect — smaller — and every row above the join is a lower bound on mixing, not a clean draw from
   $q$. This is failure (4) of the 2_2 header and it is the single thing most likely to compromise the
   leg.
2. **The free join check.** The SMC's reweighting is untouched by the `lam_base` fix
   ($d\beta \lambda' \equiv d\beta_{\mathrm{true}} \lambda_{\max}$ exactly, and the $d\beta$ bisection
   maps exactly too), so the first level of this leg tests only whether the *kernel* moved. A
   $\beta_{\mathrm{true}}$ wildly out of line with fg1's $d\beta \approx 0.10$–$0.12$ per level means
   something other than `step` changed.
3. **`accbar`, never `acc*`.** The last-sweep column exists only to show how misleading it is at
   $N = 8$; fg1's level 6 read 0.000 against a true mean of 0.133.
4. **Does $E_{q}[f]$ keep rising past $m = 2$?** fg1 got $+3.2\sigma_{p}(f)$ monotonically to
   $m_{\mathrm{eff}} = 2.00$. On the pCN line novelty is expected to rise forever, so this is *not* the
   stopping signal — degeneracy is.
5. **Do the images keep transforming or start degrading?** fg1's level 6 showed stylistic
   transformation (illustration, painterly, storybook) with latent norms drifting *down*. If norms turn
   upward past $\lambda_{\mathrm{eff}} = 157$ while `pair/iid` climbs, that is the off-manifold
   failure the readmitted band was supposed to cause and fg1 ruled out at lower tilt.

Report the level chosen by the **fixed** rule (diversity floors **and** `stop == 'decorr'`), and say
explicitly which levels the old rule would have wrongly selected.
